# Phase 1 — PDF Exploration

Goal: understand what we're dealing with before writing the ingestion pipeline.
Download a sample of arXiv papers, open them, and catalogue the types of content and parsing challenges.

In [ ]:
import arxiv
import os
from pathlib import Path

DATA_DIR = Path("../data/pdfs")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Download papers

Targeting papers that are known to have tables, figures, and equations — ML and computational biology are good bets.

In [ ]:
QUERIES = [
    "multimodal large language models benchmark",
    "protein structure prediction deep learning",
    "transformer architecture attention mechanism",
    "retrieval augmented generation evaluation",
]

client = arxiv.Client()
downloaded = []

for query in QUERIES:
    search = arxiv.Search(query=query, max_results=5, sort_by=arxiv.SortCriterion.Relevance)
    for paper in client.results(search):
        dest = DATA_DIR / f"{paper.get_short_id()}.pdf"
        if not dest.exists():
            paper.download_pdf(dirpath=str(DATA_DIR), filename=dest.name)
            print(f"Downloaded: {paper.get_short_id()} — {paper.title[:60]}")
        downloaded.append({"id": paper.get_short_id(), "title": paper.title, "path": str(dest)})

print(f"\nTotal: {len(downloaded)} papers")

## 2. Inspect with unstructured

Parse one paper and look at what element types come out.

In [ ]:
from unstructured.partition.pdf import partition_pdf
from collections import Counter

sample = next(DATA_DIR.glob("*.pdf"))
print(f"Parsing: {sample.name}")

elements = partition_pdf(
    filename=str(sample),
    strategy="hi_res",
    extract_images_in_pdf=True,
    infer_table_structure=True,
)

type_counts = Counter(type(el).__name__ for el in elements)
print("\nElement types found:")
for t, count in type_counts.most_common():
    print(f"  {t}: {count}")

In [ ]:
# Look at a few of each type
from unstructured.documents.elements import Table, Image, Title, NarrativeText

for el_type in [Title, NarrativeText, Table, Image]:
    matches = [el for el in elements if isinstance(el, el_type)]
    print(f"\n=== {el_type.__name__} ({len(matches)} found) ===")
    for el in matches[:2]:
        print(f"  page: {el.metadata.page_number}")
        print(f"  text: {str(el)[:200]}")
        print()

## 3. Known parsing challenges

Things to watch for across all papers.

In [ ]:
# Multi-page tables: check if any table spans multiple pages
tables = [el for el in elements if isinstance(el, Table)]

print(f"Tables found: {len(tables)}")
for i, t in enumerate(tables):
    text_preview = str(t)[:100].replace("\n", " ")
    print(f"  [{i}] page {t.metadata.page_number} | {text_preview}")

In [ ]:
# Equations: they typically show up as NarrativeText with garbage characters
# or as empty strings — let's find them
import re

text_elements = [el for el in elements if isinstance(el, NarrativeText)]
equation_suspects = [
    el for el in text_elements
    if re.search(r'[\\∑∫∂∇αβγδεζηθλμνξπρστφχψω]', str(el))
    or len(str(el)) < 30
]

print(f"Possible equation/garbage elements: {len(equation_suspects)}")
for el in equation_suspects[:5]:
    print(f"  page {el.metadata.page_number}: {repr(str(el)[:80])}")

In [ ]:
# Images: check if captions are detected nearby
images = [el for el in elements if isinstance(el, Image)]

print(f"Images found: {len(images)}")
for img in images[:3]:
    page = img.metadata.page_number
    # Look for caption text near the image on the same page
    same_page = [el for el in elements if el.metadata.page_number == page and el is not img]
    caption_candidates = [
        el for el in same_page
        if re.search(r'(?i)^fig(ure)?[.\s]\d', str(el))
    ]
    print(f"  page {page} | captions found: {len(caption_candidates)}")
    for cap in caption_candidates:
        print(f"    {str(cap)[:120]}")

## 4. Run across all downloaded papers

Quick pass to get aggregate stats — how many tables, images, and equation-like elements per paper.

In [ ]:
import pandas as pd

stats = []

for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    try:
        els = partition_pdf(
            filename=str(pdf_path),
            strategy="fast",  # fast pass for stats only
            infer_table_structure=False,
        )
        counts = Counter(type(el).__name__ for el in els)
        stats.append({
            "paper": pdf_path.stem,
            "total_elements": len(els),
            "tables": counts.get("Table", 0),
            "images": counts.get("Image", 0),
            "titles": counts.get("Title", 0),
            "narrative": counts.get("NarrativeText", 0),
        })
    except Exception as e:
        stats.append({"paper": pdf_path.stem, "error": str(e)})

df = pd.DataFrame(stats)
print(df.to_string())

## 5. Findings summary

Fill this in after running the cells above.

| Challenge | Observed? | Notes |
|---|---|---|
| Multi-page tables | — | — |
| Equations as garbage text | — | — |
| Long figure captions split from image | — | — |
| Section headers misclassified | — | — |
| References section polluting chunks | — | — |

**Decisions going into Phase 2:**
- Chunking strategy default: TBD after comparison
- Equation handling: filter or Nougat — TBD
- Image indexing: CLIP + caption embedding (multi-vector)